# Build a Tiny LoRA Trainer From Scratch (C++)

This notebook implements a miniature LoRA trainer using only C++.

Goals:
- Understand LoRA mathematically.
- Train low-rank adapters.
- No PEFT.
- No Transformers.
- No external files.
- Everything runs inside one notebook.

The model is intentionally tiny so the math is visible.

## LoRA Equation

Instead of training a full weight matrix W:

```text
W' = W + ΔW
```

LoRA approximates:

```text
ΔW = B × A
```

where:

- A : rank × in_features
- B : out_features × rank

Only A and B are trainable.

## Tiny Dataset

We want the model to learn:

```text
x=1 -> y=2
x=2 -> y=4
x=3 -> y=6
x=4 -> y=8
```

This simulates adapting a frozen model.

In [1]:
#include <iostream>
#include <vector>
#include <random>
#include <cmath>

struct Matrix
{
    int rows;
    int cols;
    std::vector<double> data;

    Matrix(int r, int c)
        : rows(r), cols(c), data(r*c,0.0)
    {}

    double& operator()(int r,int c)
    {
        return data[r*cols+c];
    }

    const double& operator()(int r,int c) const
    {
        return data[r*cols+c];
    }
};

Matrix matmul(const Matrix& A, const Matrix& B)
{
    Matrix C(A.rows,B.cols);

    for(int i=0;i<A.rows;i++)
        for(int j=0;j<B.cols;j++)
            for(int k=0;k<A.cols;k++)
                C(i,j)+=A(i,k)*B(k,j);

    return C;
}

void print(const Matrix& M)
{
    for(int r=0;r<M.rows;r++)
    {
        for(int c=0;c<M.cols;c++)
            std::cout << M(r,c) << " ";
        std::cout << std::endl;
    }
}

## Frozen Base Model

The frozen model weight is intentionally wrong.

Desired mapping:

```text
y = 2x
```

Base model:

```text
y = 1x
```

In [2]:
Matrix W(1,1);

W(0,0)=1.0;

std::cout<<"Base Weight\n";
print(W);

Base Weight
1 


## Create LoRA Adapter

For a 1×1 model:

rank=1

ΔW = B × A

In [3]:
std::mt19937 rng(42);
std::uniform_real_distribution<double> dist(-0.1,0.1);

Matrix A(1,1);
Matrix B(1,1);

A(0,0)=dist(rng);
B(0,0)=dist(rng);

std::cout<<"A\n";
print(A);

std::cout<<"B\n";
print(B);

A
0.0593086 
B
-0.063313 


## Training Data

In [4]:
std::vector<double> xs={1,2,3,4};
std::vector<double> ys={2,4,6,8};

## Train Only LoRA Weights

Frozen:

```text
W
```

Trainable:

```text
A
B
```

In [5]:
double lr = 0.01;

for(int epoch=0; epoch<500; epoch++)
{
    double gradA=0;
    double gradB=0;

    for(size_t i=0;i<xs.size();i++)
    {
        double delta =
            B(0,0)*A(0,0);

        double weight =
            W(0,0)+delta;

        double pred =
            weight*xs[i];

        double error =
            pred-ys[i];

        gradA +=
            2*error*xs[i]*B(0,0);

        gradB +=
            2*error*xs[i]*A(0,0);
    }

    A(0,0)-=lr*gradA;
    B(0,0)-=lr*gradB;
}

std::cout<<"Trained A\n";
print(A);

std::cout<<"Trained B\n";
print(B);

Trained A
-0.999999 
Trained B
-1 


## Compute Learned Delta Weight

In [6]:
Matrix delta = matmul(B,A);

std::cout<<"Delta W\n";
print(delta);

Matrix merged(1,1);
merged(0,0)=W(0,0)+delta(0,0);

std::cout<<"Merged Weight\n";
print(merged);

Delta W
1 
Merged Weight
2 


## Test

In [7]:
for(double x : xs)
{
    double y =
        merged(0,0)*x;

    std::cout
        << "x=" << x
        << " y=" << y
        << std::endl;
}

x=1 y=2
x=2 y=4
x=3 y=6
x=4 y=8


## Connection To Real LLMs

Real LoRA does exactly the same thing.

Instead of:

```text
1 × 1 matrix
```

it uses huge attention matrices:

```text
4096 × 4096
8192 × 8192
```

and injects:

```text
ΔW = B × A
```

into attention projections.

The core idea is identical.